# Guardrail 8 — Audit / Logging

**Where it sits:** every other guard's exit point. If a decision isn't logged, the guard didn't run.

**What it enforces:** every guard decision is recorded with provenance; PII is scrubbed *before write*; the log is append-only and queryable by request_id.

**Decision contract:** `{logged: true, request_id, line}` — this guard cannot return `block`. It either writes the entry or fails the request loud.

**Self-contained:** inlines a toy append-only log. No imports from other folders.

## Step 1 — toy append-only audit log

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


In [ ]:
import tempfile
import json, hashlib, time, re, os

PII_PATTERNS = [
    re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
    re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"),
    re.compile(r"\b(sk-[A-Za-z0-9]{20,})\b"),
]

def scrub_for_log(s: str) -> str:
    """PII scrub ON WRITE — never let raw PII touch the audit sink."""
    out = s
    for pat in PII_PATTERNS:
        out = pat.sub("[REDACTED]", out)
    return out

class AuditLog:
    SCHEMA_VERSION = 1
    def __init__(self, path: str):
        self.path = path
        # Truncate-on-open in this toy — production uses append-only storage
        # (S3 with object-lock, a WORM bucket, or a signed digest chain).
        open(self.path, "w").close()
        self._fh = open(self.path, "a", buffering=1)  # line-buffered

    def record(self, request_id: str, user_id: str, tenant_id: str,
               guard_name: str, decision: str, reasons: list,
               input_data, output_data):
        entry = {
            "schema": self.SCHEMA_VERSION,
            "ts":    time.time(),
            "request_id":  request_id,
            "user_id":     user_id,
            "tenant_id":   tenant_id,
            "guard":       guard_name,
            "decision":    decision,
            "reasons":     reasons,
            "input_hash":  hashlib.sha256(scrub_for_log(str(input_data)).encode()).hexdigest()[:16],
            "output_hash": hashlib.sha256(scrub_for_log(str(output_data)).encode()).hexdigest()[:16],
        }
        line = json.dumps(entry, separators=(",", ":"))
        self._fh.write(line + "\n")
        return entry

    def query(self, request_id: str):
        if not os.path.exists(self.path):
            return []
        out = []
        with open(self.path) as f:
            for line in f:
                try:
                    e = json.loads(line)
                    if e["request_id"] == request_id:
                        out.append(e)
                except json.JSONDecodeError:
                    pass
        return out

    def close(self):
        self._fh.close()

log = AuditLog(str(Path(tempfile.gettempdir()) / "gaurdrails-audit.log"))
print("audit log ready at", log.path)

## Step 2 — record a full request through every guard

In [ ]:
REQ = "req-2026-08-09-001"
USER = "u-alice"
TENANT = "acme"

query = "My SSN is 123-45-6789 — what is the capital of France?"

# Guard 1 — input (allow, but log PII was present)
log.record(REQ, USER, TENANT, "input_guard",
           decision="allow", reasons=["pii_present_log_scrub_required"],
           input_data=query, output_data=query)

# Guard 2 — prompt (allow)
log.record(REQ, USER, TENANT, "prompt_guard",
           decision="allow", reasons=[],
           input_data=query, output_data="<assembled safe prompt>")

# Guard 3 — retrieval (allow)
log.record(REQ, USER, TENANT, "retrieval_guard",
           decision="allow", reasons=[],
           input_data="capital of France", output_data=["d1"])

# Guard 4 — authorization (allow)
log.record(REQ, USER, TENANT, "authz_guard",
           decision="allow", reasons=[],
           input_data={"tenant":"acme","roles":["role:exec"]},
           output_data=["d1"])

# Guard 5 — document injection (allow)
log.record(REQ, USER, TENANT, "doc_injection_guard",
           decision="allow", reasons=[],
           input_data=["d1"], output_data=["d1"])

# Guard 7 — output (rewrite — PII was redacted)
log.record(REQ, USER, TENANT, "output_guard",
           decision="rewrite", reasons=["pii_redacted:1"],
           input_data="The CFO's SSN is 123-45-6789...",
           output_data="The CFO's SSN is [REDACTED]... (source: d3)")

print("recorded 6 guard decisions for", REQ)

## Step 3 — query the log

In [ ]:
entries = log.query(REQ)
print(f"query by request_id={REQ!r} → {len(entries)} entries:\n")
for e in entries:
    print(f"  [{e['guard']:20s}] {e['decision']:7s} reasons={e['reasons']}")
    print(f"     in_hash={e['input_hash']}  out_hash={e['output_hash']}")

## Step 4 — verify PII was scrubbed on write

In [ ]:
raw_log_contents = open(log.path).read()
print("raw log file contents:\n")
print(raw_log_contents)

print("\n--- PII-leak check ---")
ssn_leaked = "123-45-6789" in raw_log_contents
email_leaked = bool(re.search(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", raw_log_contents))
print(f"  SSN in log?      {ssn_leaked}  (expect False)")
print(f"  raw email in log? {email_leaked}  (expect False)")

## Step 5 — incident-response pattern: 'did guard #5 run on this request?'

In [ ]:
# An incident report says: "On req-2026-08-09-001, the model produced
# suspicious output. Did document-injection guard run?"
incident = log.query("req-2026-08-09-001")
guards_ran = [e["guard"] for e in incident]
print("guards that ran on this request:")
for g in guards_ran:
    print(f"  ✓ {g}")

if "doc_injection_guard" not in guards_ran:
    print("\nALERT: doc_injection_guard did NOT run on this request — guardrail gap.")
else:
    print("\n✓ doc_injection_guard ran — guardrail not the cause; investigate upstream.")

In [ ]:
### Real LangChain demo: the audit guard as a BaseCallbackHandler

from langchain_core.callbacks import BaseCallbackHandler

class AuditCallback(BaseCallbackHandler):
    """A minimal LangChain callback that logs every LLM/tool event."""
    def __init__(self, log):
        self.log = log
    def on_llm_start(self, serialized, prompts, **kw):
        self.log.record("lc-req", "u-alice", "acme",
                       "llm_start", "allow", [], prompts, prompts)
    def on_llm_end(self, response, **kw):
        text = str(response.generations[0][0].text) if response.generations else ""
        self.log.record("lc-req", "u-alice", "acme",
                       "llm_end", "allow", [], text, text)

if not _USE_FAKE:
    cb = AuditCallback(log)
    out = llm.invoke("What is the capital of France?", config={"callbacks": [cb]}).content
    print(f"real LLM returned: {out!r}")
    print(f"audit log entries: {len(log.query('lc-req'))}")
else:
    print("[FAKE_LLM=1 -- skipping real LLM call; callback above works the same way.] ")


## Takeaways

- **The audit log is a product surface.** Treat it like a database — schema-version it, give it a retention policy, restrict who can read it. If your security team can't query it, your guardrails are theoretical.
- **PII scrub on write, not on read.** Scrubbing at read time means every consumer needs to remember to scrub. Scrubbing at write time means the sink is always clean.
- **Hash inputs and outputs, don't store them.** The audit log doesn't need the user's query; it needs to prove *what* flowed through. A SHA-256 prefix is enough to correlate 'same input' across entries without storing PII.
- **Append-only or it isn't an audit log.** If an attacker can edit a line, the log is a confession they can rewrite. Use object-lock, WORM buckets, or signed digest chains.
- **Use the log to prove guardrails ran.** 'Did guard #5 run on this request?' should be a one-line query, not a code archaeology expedition. If it isn't, the log isn't doing its job.
- **Alert on guard-regression.** If guard #5's block rate drops from 3% to 0.1% overnight, either the attack stopped (good) or the guard broke (bad). The log is what tells you which.

**Negative fixture checklist:** scrubbed PII never appears on disk; every block decision has a matching log entry; query-by-request-id returns the full guard chain. ✓